# 03 · RQ2: classical machine learning versus a lightweight CNN

Trains LR, SVM, RF and XGBoost on the 64 features (tuned on the validation subset) and a 4-block CNN on log-mel spectrograms, then compares them on **Test-Known** (held-out speakers, known generation algorithms).

H₀: the CNN and the best classical model have equal accuracy on paired clips. H₁: they differ (McNemar test).

Use a GPU runtime for the CNN (Runtime → Change runtime type → GPU).

In [ ]:
# --- Setup: works on Google Colab and locally -------------------------------
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ.setdefault("VS_DATA_ROOT", "/content/drive/MyDrive/audio-detection-human-vs-ai")
    if not os.path.exists("/content/audio-detection-human-vs-ai"):
        !git clone -q https://github.com/abhilash-u/audio-detection-human-vs-ai /content/audio-detection-human-vs-ai
    %cd /content/audio-detection-human-vs-ai
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np, pandas as pd
from src import config as C
C.ensure_dirs()
print("Data folder:", C.DATA_ROOT)

In [ ]:
import time, joblib
from src import data, evaluation as E, models as M
from src.features import FEATURES
sample = pd.read_csv(C.META_DIR / "split_assignments.csv")
X = pd.read_csv(C.FEAT_DIR / "features_clean.csv")
L = np.load(C.FEAT_DIR / "logmel_clean.npy")
df = sample.merge(X, on="clip_id")          # same row order as L
idx = {s: np.where(df.split == s)[0] for s in df.split.unique()}
tr, va = idx["training"], idx["validation"]
tk = np.r_[idx["test_human"], idx["test_known"]]
y = df.label.values

## Classical models

In [ ]:
results, scores, fitted = [], {}, {}
for name in ["LR", "SVM", "RF", "XGBoost"]:
    t0 = time.time()
    pipe, params, v_eer = M.tune_classical(name, df.loc[tr, FEATURES], y[tr], df.loc[va, FEATURES], y[va])
    train_s = time.time() - t0
    thr = E.eer(y[va], pipe.predict_proba(df.loc[va, FEATURES])[:, 1])[1]   # threshold fixed on validation
    t0 = time.time(); s = pipe.predict_proba(df.loc[tk, FEATURES])[:, 1]; infer_ms = 1000 * (time.time() - t0) / len(tk)
    scores[name], fitted[name] = s, (pipe, thr)
    results.append({"model": name, "params": params, "val_eer": v_eer, "threshold": thr,
                    **E.classification_report(y[tk], s, thr), "train_s": train_s, "infer_ms_per_clip": infer_ms})
    joblib.dump({"model": pipe, "threshold": thr}, C.MODEL_DIR / f"{name}.joblib")
pd.DataFrame(results).drop(columns="params").round(4)

## Learning curves (is the training set large enough?)

In [ ]:
best_name = min(results, key=lambda r: r["val_eer"])["model"]
rng = np.random.default_rng(C.SEED); lc = []
for frac in (0.1, 0.25, 0.5, 0.75, 1.0):
    sub = rng.choice(tr, size=int(frac * len(tr)), replace=False)
    pipe, _, v = M.tune_classical(best_name, df.loc[sub, FEATURES], y[sub], df.loc[va, FEATURES], y[va])
    lc.append({"fraction": frac, "n_train": len(sub), "val_eer": v})
lc = pd.DataFrame(lc); lc.to_csv(C.REPORT_DIR / "rq2_learning_curve.csv", index=False); lc

## Lightweight CNN on log-mel spectrograms

In [ ]:
t0 = time.time()
cnn, mu, sd, hist = M.train_cnn(L[tr], y[tr], L[va], y[va], epochs=30)
cnn_train_s = time.time() - t0
cnn_thr = E.eer(y[va], M.predict_cnn(cnn, L[va], mu, sd))[1]
t0 = time.time(); s_cnn = M.predict_cnn(cnn, L[tk], mu, sd); cnn_ms = 1000 * (time.time() - t0) / len(tk)
import torch
torch.save({"state_dict": cnn.state_dict(), "mean": mu, "std": sd, "threshold": cnn_thr}, C.MODEL_DIR / "CNN.pt")
scores["CNN"] = s_cnn
results.append({"model": "CNN", "val_eer": min(h["val_eer"] for h in hist), "threshold": cnn_thr,
                **E.classification_report(y[tk], s_cnn, cnn_thr), "train_s": cnn_train_s, "infer_ms_per_clip": cnn_ms})
res = pd.DataFrame(results).drop(columns="params", errors="ignore")
res.to_csv(C.REPORT_DIR / "rq2_test_known_metrics.csv", index=False); res.round(4)

## Speaker-cluster bootstrap CIs and McNemar test (CNN vs best classical)

In [ ]:
test_df = df.loc[tk, ["speaker_id", "label"]].reset_index(drop=True)
ci_rows = []
for name, s in scores.items():
    t = test_df.assign(s=s)
    for metric, fn in {"roc_auc": lambda d: E.roc_auc_score(d.label, d.s), "eer": lambda d: E.eer(d.label.values, d.s.values)[0]}.items():
        est, lo, hi = E.cluster_bootstrap(t, fn, n_boot=1000)
        ci_rows.append({"model": name, "metric": metric, "estimate": est, "ci_low": lo, "ci_high": hi})
ci = pd.DataFrame(ci_rows); ci.to_csv(C.REPORT_DIR / "rq2_bootstrap_ci.csv", index=False)
best_classical = min([r for r in results if r["model"] != "CNN"], key=lambda r: r["val_eer"])["model"]
thr_c = fitted[best_classical][1]
correct_c = (scores[best_classical] >= thr_c).astype(int) == y[tk]
correct_n = (scores["CNN"] >= cnn_thr).astype(int) == y[tk]
mc = E.mcnemar(correct_n, correct_c)
print(f"CNN vs {best_classical}: b={mc['b']} c={mc['c']} statistic={mc['statistic']:.3f} p={mc['p_value']:.4f} ({mc['method']})")
display(ci.pivot(index="model", columns="metric", values=["estimate", "ci_low", "ci_high"]).round(4))

## ROC curves and permutation importance (feeds back into RQ1)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
from sklearn.inspection import permutation_importance
fig, ax = plt.subplots(figsize=(5.5, 5))
for name, s in scores.items():
    f, t, _ = roc_curve(y[tk], s); ax.plot(f, t, label=name)
ax.plot([0, 1], [0, 1], "k--", lw=0.8); ax.set_xlabel("False-positive rate"); ax.set_ylabel("True-positive rate")
ax.legend(); ax.set_title("RQ2: ROC on Test-Known"); plt.tight_layout(); plt.savefig(C.REPORT_DIR / "rq2_roc.png", dpi=200); plt.show()
pi = permutation_importance(fitted[best_classical][0], df.loc[va, FEATURES], y[va], scoring="roc_auc", n_repeats=5, random_state=C.SEED)
imp = pd.DataFrame({"feature": FEATURES, "importance": pi.importances_mean}).sort_values("importance", ascending=False)
imp.to_csv(C.REPORT_DIR / "rq1_permutation_importance.csv", index=False); imp.head(15)

In [ ]:
pd.to_pickle({"scores": scores, "thresholds": {**{k: v[1] for k, v in fitted.items()}, "CNN": cnn_thr},
              "best_classical": best_classical}, C.MODEL_DIR / "rq2_state.pkl")